# TinyYOLO Experiment Suite — Notebook 5/7
## Multi-Task Validation (Table VIII)

**What:** Train segmentation (TinySegment) and pose (TinyPose) on COCO.

**GPU Time:** ~12h T4 (2 tasks × ~6h each)

In [ ]:
#@title ⚙️ Configuration — Edit this cell
FAST_MODE = True  #@param {type:"boolean"}

if FAST_MODE:
    print("🚀 Running in FAST_MODE (Rapid Verification)")
    EPOCHS = 50                  # Faster multi-task validation
    IMGSZ = 320                  # Reduced resolution
    BATCH = 64
else:
    print("📚 Running in FULL PUBLICATION RIGOR mode")
    EPOCHS = 150
    IMGSZ = 416
    BATCH = 64

In [ ]:
import os, sys, socket, shutil
from pathlib import Path
REPO = 'https://github.com/ShMazumder/tinyYOLO.git'
try: socket.create_connection(('github.com', 80), timeout=5)
except: print('❌ No internet!'); sys.exit(1)
if Path('tinyYOLO').exists(): shutil.rmtree('tinyYOLO')
!git clone {REPO} --depth 1
%cd tinyYOLO
!pip install -e . -q 2>&1 | tail -1
!pip install tqdm timm psutil -q 2>&1 | tail -1
import torch
if torch.cuda.is_available(): print(f'🔥 GPU: {torch.cuda.get_device_name(0)}')
print('✅ Setup complete!')

In [ ]:
# Instance Segmentation — TinySegment on COCO
!python scripts/train.py --task seg --variant quantized --data coco128-seg.yaml --lr 2e-4 --amp --cache --freeze-epochs 5 --grad-clip 5.0 --val-conf 0.001 --ema-decay 0.9998 --close-mosaic 40 \
    --imgsz {IMGSZ} --epochs {EPOCHS} --batch {BATCH} --seed 42 --warmup 3 \
    --pretrained --compile --name multitask-seg-q

!python scripts/train.py --task seg --variant standard --data coco128-seg.yaml --lr 2e-4 --amp --cache --freeze-epochs 5 --grad-clip 5.0 --val-conf 0.001 --ema-decay 0.9998 --close-mosaic 40 \
    --imgsz {IMGSZ} --epochs {EPOCHS} --batch {BATCH} --seed 42 --warmup 3 \
    --pretrained --compile --name multitask-seg-s

In [ ]:
# Pose Estimation — TinyPose on COCO
!python scripts/train.py --task pose --variant quantized --data coco8-pose.yaml --lr 2e-4 --amp --cache --freeze-epochs 5 --grad-clip 5.0 --val-conf 0.001 --ema-decay 0.9998 --close-mosaic 40 \
    --imgsz {IMGSZ} --epochs {EPOCHS} --batch {BATCH} --seed 42 --warmup 3 \
    --pretrained --compile --name multitask-pose-q

!python scripts/train.py --task pose --variant standard --data coco8-pose.yaml --lr 2e-4 --amp --cache --freeze-epochs 5 --grad-clip 5.0 --val-conf 0.001 --ema-decay 0.9998 --close-mosaic 40 \
    --imgsz {IMGSZ} --epochs {EPOCHS} --batch {BATCH} --seed 42 --warmup 3 \
    --pretrained --compile --name multitask-pose-s

In [ ]:
import json
from pathlib import Path

print('='*70)
print('MULTI-TASK RESULTS')
print('='*70)
for name in ['multitask-seg-q', 'multitask-seg-s', 'multitask-pose-q', 'multitask-pose-s']:
    s = Path(f'experiments/results/{name}/summary.json')
    if s.exists():
        with open(s) as f: data = json.load(f)
        print(f'  {name}: {json.dumps({k:v for k,v in data.items() if "map" in k.lower() or "loss" in k.lower()}, indent=2)}')
    else:
        print(f'  {name}: N/A')
print('='*70)